# معمل التجارب: بناء مسار RAG متكامل مع التقطيع الدلالي المعياري (المحاضرة 39)
## القسم الأول: الشرح النظري والمعماري

في هذا المعمل العملي، ننظم عملية التقطيع الدلالي ككود إنتاجي معياري (Modular OOP):
1. بناء كلاس `ThresholdSemanticChunker` لتقطيع النصوص والوثائق مع الحفاظ على البيانات الوصفية.
2. فهرسة القطع الدلالية في مخزن المتجهات `FAISS`.
3. تجميع مسار RAG كامل عبر تعبيرات `LCEL` مع موديل `Groq` اللغوي.


## القسم الثاني: الكود المرجعي للتطبيق والتجربة
شغل الخلايا التالية لمتابعة تنفيذ المسار كاملاً، ثم استخدم مساحة التجربة الحرة في الأسفل للتعديل.

In [1]:
# 1. استدعاء المكتبات وتهيئة البيئة
import os
import warnings
from typing import List
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

print(f"[CONFIG] GROQ_API_KEY ready: {bool(os.getenv('GROQ_API_KEY'))}")

[CONFIG] GROQ_API_KEY ready: True


In [2]:
# 2. كلاس التقطيع المعياري
class ThresholdSemanticChunker:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", threshold: float = 0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split_text(self, text: str) -> List[str]:
        sentences = [s.strip() for s in text.split("\n") if s.strip()]
        if not sentences:
            return []
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]
        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks

    def split_documents(self, docs: List[Document]) -> List[Document]:
        result = []
        for doc in docs:
            chunks = self.split_text(doc.page_content)
            for chunk in chunks:
                result.append(Document(page_content=chunk, metadata=doc.metadata.copy()))
        return result

print("[INFO] Chunker class initialized.")

[INFO] Chunker class initialized.


In [3]:
# 3. إعداد الوثيقة وتشغيل المسار الكامل
sample_text = """LangChain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

raw_doc = Document(page_content=sample_text, metadata={"source": "playground_test"})
chunker = ThresholdSemanticChunker(threshold=0.7)
doc_chunks = chunker.split_documents([raw_doc])

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(doc_chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

prompt = ChatPromptTemplate.from_template("""Answer based only on the context:
Context: {context}
Question: {question}
Answer:""")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever,
        question=lambda x: x["question"]
    )
    | prompt
    | llm
    | StrOutputParser()
)

result = rag_chain.invoke({"question": "What is LangChain used for?"})
print(f"[RESULT]: {result}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[RESULT]: LangChain is a framework for building applications that use large language models (LLMs). It offers modular abstractions so developers can combine LLMs with tools such as OpenAI, Pinecone, and others, and create components like chains, agents, memory, and retrievers.


---
## القسم الثالث: مساحة التجربة العملية الخاصة بك
اكتب كودك هنا لتجربة نصوص مخصصة، أو تغيير بارامترات التقطيع، أو اختبار أسئلة جديدة.

In [4]:
# مساحتك للتجربة والتطبيق العملي:
# جرب استدعاء الـ chunker بعتبة مختلفة أو نص جديد من عندك وشغله هنا:

